<a href="https://colab.research.google.com/github/MawaSC/Quantum-Simulation-on-IQM-Resonance/blob/main/01_fermi_hubbard_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Model and system parameters

We study a one-dimensional Fermi–Hubbard chain with L lattice sites.
Each site has two spin orbitals, spin-up and spin-down, represented
by 2L qubits.

In [ ]:
%pip install -U qrisp numpy scipy matplotlib
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from functools import reduce

from qrisp import QuantumVariable, x, h, rx, rz, rzz, swap, cz

np.set_printoptions(precision=4, suppress=True, linewidth=150)
print("qrisp imported")

The model includes particles moving between neighboring sites, an interaction between particles on the same site ((U)), and a chemical potential ((\mu)). The hopping strength ((t_h)) sets the energy scale, so time is measured in units of (1/t_h).

The parameters below define the system used for the initial tests. In later experiments, we may change the timestep while keeping the total simulation time the same.

In [ ]:
L      = 4      # number of lattice sites
N_Q    = 2 * L  # number of qubits (one per spin-orbital)
t_h    = 1.0    # hopping amplitude  (sets the unit of time)
U      = 4.0    # on-site repulsion
mu     = 0.0    # chemical potential

print(f"L = {L} sites  ->  {N_Q} qubits,  Hilbert space dim = {2**N_Q}")
print(f"U/t_h = {U/t_h}")

2. Validate the Jordan–Wigner Hamiltonian

We build the Hamiltonian in two different ways: first using fermionic creation and annihilation operators, and then using Pauli operators after applying the Jordan–Wigner mapping.

We compare the resulting matrices to check that both methods give the same Hamiltonian with our chosen qubit ordering. This is an important first step because any mistakes here could affect all the simulations that follow.

If the matrices do not match within the chosen tolerance, the notebook should raise an error so we can fix the issue before moving on.

In [ ]:
def qubit_of(site, spin, flipped=False):
    """Qubit index holding spin-orbital (site, spin) under the pair-interleaved ordering.

    spin    : 'u' or 'd'
    flipped : True after an odd number of fSWAP layers, when the two orbitals
              of every site have exchanged qubits.
    """
    first = 'd' if site % 2 == 0 else 'u'   # which spin sits on the lower-index qubit
    offset = 0 if spin == first else 1
    if flipped:
        offset = 1 - offset
    return 2 * site + offset


print("qubit :  (site, spin)")
for J in range(N_Q):
    site = J // 2
    lbl = [s for s in ('u', 'd') if qubit_of(site, s) == J][0]
    arrow = "up" if lbl == 'u' else "down"
    print(f"   q{J}   :   site {site}, spin {arrow}")

3. Validate elementary gates

The hopping part of the circuit is built using XX and YY rotations. Before using these gates in the full circuit, we compare them with the results we get by calculating the same operations directly using matrices.

We also test the fermionic SWAP gate, which switches two orbitals while adding the required minus sign when both are occupied. These tests help us catch mistakes with signs, rotation angles, or gate order before running the full simulation.

In [ ]:
# Where do the Hamiltonian terms land?
print("on-site U terms  (R_ZZ) act on qubit pairs:")
print("   ", [(2*i, 2*i+1) for i in range(L)])

print("\nshort hops  XX+YY  (weight 2) act on qubit pairs:")
print("   ", [(2*j+1, 2*j+2) for j in range(L-1)])

print("\nlong hops  XZZX+YZZY  (weight 4) act on qubit quadruples:")
print("   ", [(2*j, 2*j+1, 2*j+2, 2*j+3) for j in range(L-1)])

print("\ninterpretation:")
for j in range(L-1):
    a, b = 2*j+1, 2*j+2
    sa = [s for s in 'ud' if qubit_of(a//2, s) == a][0]
    print(f"   short hop (q{a},q{b}) = site {a//2} <-> site {b//2}, spin {'up' if sa=='u' else 'down'}")
for j in range(L-1):
    a, d = 2*j, 2*j+3
    sa = [s for s in 'ud' if qubit_of(a//2, s) == a][0]
    print(f"   long  hop (q{a},q{d}) = site {a//2} <-> site {d//2}, spin {'up' if sa=='u' else 'down'}")

In [ ]:
#Matrices
I2 = np.eye(2)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]])
Z  = np.diag([1, -1]).astype(complex)

def op(n, terms):
    """Tensor product over n qubits; qubit 0 is the leftmost (most significant) factor."""
    return reduce(np.kron, [terms.get(k, I2) for k in range(n)])

def jw_annihilate(n, J):
    """c_J under Jordan-Wigner: Z-string on all k < J."""
    terms = {k: Z for k in range(J)}
    terms[J] = np.array([[0, 1], [0, 0]], dtype=complex)   # |0><1|
    return op(n, terms)


def H_fermionic(L, t_h, U, mu):
    """Build H from fermionic operators, JW-mapped with the pair-interleaved ordering."""
    n = 2 * L
    c = {(i, s): jw_annihilate(n, qubit_of(i, s)) for i in range(L) for s in 'ud'}
    H = np.zeros((2**n, 2**n), dtype=complex)
    for i in range(L - 1):                       # hopping
        for s in 'ud':
            a, b = c[(i, s)], c[(i+1, s)]
            H += -t_h * (a.conj().T @ b + b.conj().T @ a)
    for i in range(L):                           # interaction + chemical potential
        nu = c[(i, 'u')].conj().T @ c[(i, 'u')]
        nd = c[(i, 'd')].conj().T @ c[(i, 'd')]
        H += U * (nu @ nd) - mu * (nu + nd)
    return H


def H_qubitized(L, t_h, U, mu):
    """Eq. (6) of the paper, written directly as Pauli strings."""
    n = 2 * L
    H = np.zeros((2**n, 2**n), dtype=complex)
    for j in range(L - 1):
        H += -t_h/2 * (op(n, {2*j+1: X, 2*j+2: X}) + op(n, {2*j+1: Y, 2*j+2: Y}))
        H += -t_h/2 * (op(n, {2*j: X, 2*j+1: Z, 2*j+2: Z, 2*j+3: X})
                     + op(n, {2*j: Y, 2*j+1: Z, 2*j+2: Z, 2*j+3: Y}))
    Id = np.eye(2**n)
    for i in range(L):
        H += U/4 * (Id - op(n, {2*i: Z})) @ (Id - op(n, {2*i+1: Z}))
    for J in range(n):
        H += -mu/2 * (Id - op(n, {J: Z}))
    return H


for test_mu in (0.0, 0.7):
    A = H_fermionic(L, t_h, U, test_mu)
    B = H_qubitized(L, t_h, U, test_mu)
    print(f"mu={test_mu}:  Eq.(6) matches direct JW?  {np.allclose(A, B)}"
          f"   (max |difference| = {np.abs(A-B).max():.2e})")

In [ ]:
def rxx_(theta, a, b):
    """exp(-i theta/2 * X⊗X)"""
    h(a); h(b); rzz(theta, a, b); h(a); h(b)

def ryy_(theta, a, b):
    """exp(-i theta/2 * Y⊗Y)"""
    rx(np.pi/2, a);  rx(np.pi/2, b)
    rzz(theta, a, b)
    rx(-np.pi/2, a); rx(-np.pi/2, b)

def hop(dt, t_h, a, b):
    """exp(+i t_h dt/2 (XX+YY)) : one hopping bond for time dt"""
    rxx_(-t_h * dt, a, b)
    ryy_(-t_h * dt, a, b)

def fswap(a, b):
    """fermionic SWAP = SWAP followed by CZ"""
    swap(a, b)
    cz(a, b)


# --- sanity check the gate conventions against explicit matrix exponentials ---
def unitary_of(n, builder):
    qv = QuantumVariable(n); builder(qv); return qv.qs.compile().get_unitary()

th = 0.63
chk = {
 "R_XX": (unitary_of(2, lambda q: rxx_(th, q[0], q[1])), la.expm(-1j*th/2*np.kron(X, X))),
 "R_YY": (unitary_of(2, lambda q: ryy_(th, q[0], q[1])), la.expm(-1j*th/2*np.kron(Y, Y))),
 "hop ": (unitary_of(2, lambda q: hop(0.3, 1.0, q[0], q[1])),
          la.expm(1j*0.3/2*(np.kron(X, X) + np.kron(Y, Y)))),
 "fSWAP": (unitary_of(2, lambda q: fswap(q[0], q[1])),
           np.array([[1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,-1]], dtype=complex)),
}
# Qrisp's simulator works in single precision, so compare at ~1e-5, not 1e-8.
for name, (got, want) in chk.items():
    print(f"  {name:6s} correct? {np.allclose(got, want, atol=1e-5)}"
          f"   (max |difference| = {np.abs(got-want).max():.1e})")

In [ ]:
def layer_rz(qv, dt):
    theta = (mu - U/2) * dt
    if abs(theta) > 1e-15:
        for J in range(N_Q):
            rz(theta, qv[J])

def layer_rzz(qv, dt):
    for i in range(L):
        rzz(U * dt / 2, qv[2*i], qv[2*i+1])

def layer_hop(qv, dt):
    for j in range(L - 1):
        hop(dt, t_h, qv[2*j+1], qv[2*j+2])

def layer_fswap(qv):
    for i in range(L):
        fswap(qv[2*i], qv[2*i+1])


def trotter_step(qv, dt):
    layer_rz(qv, dt/2)     # purple
    layer_hop(qv, dt)      # blue   - short hops
    layer_rzz(qv, dt)      # yellow - on-site U
    layer_fswap(qv)        # green  - spin exchange
    layer_hop(qv, dt)      # blue   - the other half of the hops
    layer_rz(qv, dt/2)     # purple


def build_circuit(dt, n_steps, init_occ):
    """Prepare a Fock state, then apply n_steps Trotter layers."""
    qv = QuantumVariable(N_Q)
    for J, occupied in enumerate(init_occ):
        if occupied:
            x(qv[J])
    for _ in range(n_steps):
        trotter_step(qv, dt)
    return qv


qv_demo = build_circuit(0.2, 1, np.zeros(N_Q, dtype=int))
qc = qv_demo.qs.compile()
print(f"one Trotter step, L={L}:  depth = {qc.depth()},  {len(qc.data)} gates")

In [ ]:
def neel_state(L, vacancy=None):
    """Half-filled Neel state, optionally with one site emptied."""
    occ = np.zeros(2*L, dtype=int)
    for i in range(L):
        if vacancy is not None and i == vacancy:
            continue
        spin = 'd' if i % 2 == 0 else 'u'
        occ[qubit_of(i, spin)] = 1
    return occ


def occupations_from_counts(counts, flipped):
    """counts: {bitstring: probability} from qrisp. Returns n_up[L], n_down[L]."""
    p1 = np.zeros(N_Q)
    for bits, p in counts.items():
        for J, b in enumerate(bits):
            if b == '1':
                p1[J] += p
    total = sum(counts.values())
    p1 /= total
    n_up   = np.array([p1[qubit_of(i, 'u', flipped)] for i in range(L)])
    n_down = np.array([p1[qubit_of(i, 'd', flipped)] for i in range(L)])
    return n_up, n_down


init_occ = neel_state(L)
print("initial qubit occupations:", init_occ)
nu, nd = occupations_from_counts({''.join(map(str, init_occ)): 1.0}, flipped=False)
print("n_up   per site:", nu)
print("n_down per site:", nd)
print("filling n = N/L  =", (nu.sum() + nd.sum()) / L)